# IP_GFP + GV_GFP UMAPs for prioritized programs and Cdh5

This notebook replaces the discrete dominant-program plot with **continuous usage UMAPs**.

It produces:

1. a four-panel UMAP for **P15, P9, P6, and P12**;
2. one individual UMAP for each program;
3. a UMAP of normalized **Cdh5** expression;
4. a five-panel overview containing the four programs plus Cdh5;
5. a cell-level CSV containing UMAP coordinates, all four program usages, and Cdh5 expression.

No program labels are placed over the cells. Panel titles appear above each axis.

Program color represents continuous cNMF usage in every cell, not whether the program is globally dominant. Values above the configured quantile are clipped only for visualization; exported CSV values are unmodified.

In [ ]:
# Set thread limits before importing NumPy/SciPy.
import os

for variable in [
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "BLIS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
]:
    os.environ.setdefault(variable, "1")

os.environ.setdefault("PYTHONNOUSERSITE", "1")


In [ ]:
from __future__ import annotations

from pathlib import Path
import re
import sys
import warnings

import anndata as ad
from IPython.display import display
import matplotlib as mpl
import matplotlib.pyplot as plt
import muon as mu
import numpy as np
import pandas as pd
import scipy.sparse as sp

warnings.filterwarnings("ignore")

# Keep text and plotted points editable in PDF/SVG.
mpl.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 8,
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.03,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    }
)

print("Python:", sys.version.split()[0])


## Configuration

In [ ]:
# ---------------------------------------------------------------------
# Selected K30 / density-threshold-2.0 model
# ---------------------------------------------------------------------
BASE = Path(
    "/oak/stanford/groups/engreitz/Users/tri/10xPerturbSeq/"
    "analysis/250507_ePAH1_NovaSeq/CRISPR_count/"
    "IP_GV_unified_sceptre_cnmf_no_celltype"
)

RUN_NAME = "IP_GV_skcnmf_K30_dt2_0_n100"
RUN_ROOT = BASE / "PerturbNMF" / "results" / RUN_NAME

EXPECTED_H5MU_PATH = (
    RUN_ROOT
    / "Inference"
    / "adata"
    / "cNMF_30_2_0.h5mu"
)

if EXPECTED_H5MU_PATH.exists():
    H5MU_PATH = EXPECTED_H5MU_PATH
else:
    candidates = sorted(
        (BASE / "PerturbNMF").glob(
            "**/Inference/adata/cNMF_30_2_0.h5mu"
        )
    )
    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Expected {EXPECTED_H5MU_PATH}, and auto-discovery found "
            f"{len(candidates)} candidates: {candidates}"
        )
    H5MU_PATH = candidates[0]
    RUN_ROOT = H5MU_PATH.parents[2]

# Fallback UMAP source if X_umap is not retained in the compiled H5MU.
PREPARED_H5AD_PATH = (
    BASE
    / "PerturbNMF"
    / "input"
    / "IP_GFP_GV_GFP_guide_positive_PerturbNMF_input.h5ad"
)

OUTPUT_DIR = (
    RUN_ROOT
    / "Evaluation"
    / "30_2_0"
    / "selected_program_umap"
    / "continuous_usage_and_Cdh5"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# Programs and labels
# ---------------------------------------------------------------------
SELECTED_PROGRAMS = [15, 9, 6, 12]

PROGRAM_LABELS = {
    15: "Homeostatic EC",
    9: "Angiogenic migration",
    6: "Inflammatory adhesion",
    12: "sGC-cGMP remodeling",
}

MARKER_GENE = "Cdh5"

# ---------------------------------------------------------------------
# Plot settings
# ---------------------------------------------------------------------
PROGRAM_CMAP = "viridis"
GENE_CMAP = "magma"

# The top 1% is clipped in the plots so a few extreme cells do not compress
# the color range. Raw values remain unchanged in the exported CSV.
COLOR_CLIP_QUANTILE = 0.99

BACKGROUND_COLOR = "#DDE1E6"
BACKGROUND_ALPHA = 0.55
BACKGROUND_SIZE = 5.0

POINT_SIZE = 8.0
POINT_ALPHA = 0.95
RASTERIZE_POINTS = False  # False keeps points vector-editable in PDF/SVG.

PROGRAM_FIGSIZE = (8.6, 7.4)
INDIVIDUAL_FIGSIZE = (4.6, 4.1)
GENE_FIGSIZE = (4.8, 4.2)
OVERVIEW_FIGSIZE = (12.0, 7.6)

OUTPUT_STEM = "IP_GV_P15_P9_P6_P12_continuous_usage_Cdh5"

print("H5MU:", H5MU_PATH)
print("Prepared H5AD fallback:", PREPARED_H5AD_PATH)
print("Output directory:", OUTPUT_DIR)


## Helper functions

In [ ]:
def dense_array(matrix) -> np.ndarray:
    if sp.issparse(matrix):
        return matrix.toarray()
    if hasattr(matrix, "toarray"):
        return matrix.toarray()
    return np.asarray(matrix)


def infer_program_ids(var_names: pd.Index) -> list[int]:
    """Infer one-based program numbers from cNMF variable names."""
    parsed = []
    for value in var_names.astype(str):
        match = re.search(r"(\d+)$", value)
        parsed.append(int(match.group(1)) if match else None)

    n_programs = len(var_names)

    if all(value is not None for value in parsed):
        parsed_int = [int(value) for value in parsed]
        if len(set(parsed_int)) == n_programs:
            if set(parsed_int) == set(range(n_programs)):
                return [value + 1 for value in parsed_int]
            return parsed_int

    print(
        "Could not infer unique program IDs from var_names; "
        "using column order as P1..PK."
    )
    return list(range(1, n_programs + 1))


def align_obsm_to_target(
    source,
    target_obs_names: pd.Index,
    key: str = "X_umap",
) -> np.ndarray:
    if key not in source.obsm:
        raise KeyError(f"{key!r} is absent from source.obsm.")

    source_index = pd.Index(source.obs_names)
    target_index = pd.Index(target_obs_names)

    if not source_index.is_unique or not target_index.is_unique:
        raise ValueError("Cell indexes must be unique.")

    missing = target_index.difference(source_index)
    if len(missing):
        raise ValueError(
            f"{len(missing)} target cells are absent from the UMAP source. "
            f"Examples: {missing[:10].tolist()}"
        )

    positions = source_index.get_indexer(target_index)
    coordinates = np.asarray(source.obsm[key])[positions, :2]

    if coordinates.shape != (len(target_index), 2):
        raise ValueError(
            f"Aligned UMAP has unexpected shape {coordinates.shape}."
        )
    return coordinates


def resolve_umap(
    *,
    mdata,
    cnmf,
    rna,
    prepared_h5ad_path: Path,
) -> tuple[np.ndarray, str]:
    if "X_umap" in cnmf.obsm:
        return (
            align_obsm_to_target(
                cnmf,
                pd.Index(cnmf.obs_names),
                "X_umap",
            ),
            "cNMF.obsm['X_umap']",
        )

    if "X_umap" in rna.obsm:
        return (
            align_obsm_to_target(
                rna,
                pd.Index(cnmf.obs_names),
                "X_umap",
            ),
            "rna.obsm['X_umap']",
        )

    if "X_umap" in mdata.obsm:
        coordinates = np.asarray(mdata.obsm["X_umap"])
        if coordinates.shape[0] != cnmf.n_obs:
            raise ValueError(
                "MuData X_umap does not have the expected number of cells."
            )
        return coordinates[:, :2], "mdata.obsm['X_umap']"

    if not prepared_h5ad_path.exists():
        raise FileNotFoundError(
            "No X_umap was found in the H5MU and the prepared H5AD "
            f"does not exist: {prepared_h5ad_path}"
        )

    prepared = ad.read_h5ad(prepared_h5ad_path, backed="r")
    try:
        coordinates = align_obsm_to_target(
            prepared,
            pd.Index(cnmf.obs_names),
            "X_umap",
        )
    finally:
        prepared.file.close()

    return coordinates, str(prepared_h5ad_path)


def matrix_is_integer_like(
    matrix,
    tolerance: float = 1e-3,
    maximum_values: int = 200_000,
) -> bool:
    if sp.issparse(matrix):
        values = np.asarray(matrix.data)
    else:
        values = np.asarray(matrix).ravel()

    values = values[np.isfinite(values)]
    if values.size == 0:
        return True

    if values.size > maximum_values:
        positions = np.linspace(
            0,
            values.size - 1,
            maximum_values,
            dtype=int,
        )
        values = values[positions]

    if values.min() < 0:
        return False

    return bool(
        np.max(np.abs(values - np.rint(values))) <= tolerance
    )


def resolve_expression_source(adata):
    """Return matrix, var table, var names, source name, and count status."""
    candidates = []

    for layer_name in ["counts", "raw_counts", "count"]:
        if layer_name in adata.layers:
            candidates.append(
                (
                    adata.layers[layer_name],
                    adata.var,
                    pd.Index(adata.var_names),
                    f"rna.layers[{layer_name!r}]",
                )
            )

    candidates.append(
        (
            adata.X,
            adata.var,
            pd.Index(adata.var_names),
            "rna.X",
        )
    )

    if adata.raw is not None:
        candidates.append(
            (
                adata.raw.X,
                adata.raw.var,
                pd.Index(adata.raw.var_names),
                "rna.raw.X",
            )
        )

    for matrix, var, var_names, source_name in candidates:
        if matrix_is_integer_like(matrix):
            return matrix, var, var_names, source_name, True

    matrix, var, var_names, source_name = candidates[0]
    return matrix, var, var_names, source_name, False


def find_gene_columns(
    var: pd.DataFrame,
    var_names: pd.Index,
    gene: str,
) -> list[int]:
    target = gene.strip().upper()
    candidate_series = []

    for column in [
        "symbol",
        "feature_name",
        "gene_symbol",
        "gene_name",
    ]:
        if column in var.columns:
            candidate_series.append(var[column].astype(str))

    candidate_series.append(
        pd.Series(var_names.astype(str), index=var_names)
    )

    matched = set()
    for series in candidate_series:
        values = series.astype(str).str.strip().str.upper().to_numpy()
        matched.update(np.flatnonzero(values == target).tolist())

    return sorted(matched)


def extract_gene_expression(
    adata,
    gene: str,
) -> tuple[np.ndarray, str, list[str]]:
    matrix, var, var_names, source_name, is_counts = (
        resolve_expression_source(adata)
    )

    columns = find_gene_columns(var, var_names, gene)
    if not columns:
        raise KeyError(
            f"Could not find {gene!r} in RNA feature symbols or names."
        )

    if sp.issparse(matrix):
        gene_values = np.asarray(
            matrix[:, columns].sum(axis=1)
        ).ravel()
    else:
        gene_values = np.asarray(
            matrix[:, columns]
        ).sum(axis=1)

    if is_counts:
        if sp.issparse(matrix):
            library_size = np.asarray(
                matrix.sum(axis=1)
            ).ravel()
        else:
            library_size = np.asarray(matrix).sum(axis=1)

        normalized = np.divide(
            1e4 * gene_values,
            library_size,
            out=np.zeros_like(gene_values, dtype=float),
            where=library_size > 0,
        )
        expression = np.log1p(normalized)
        label = f"log1p normalized {gene} expression"
    else:
        expression = np.asarray(gene_values, dtype=float)
        label = f"{gene} expression from {source_name}"

    matched_features = var_names[np.asarray(columns)].astype(str).tolist()
    return expression, label, matched_features


def robust_vmax(
    values: np.ndarray,
    quantile: float = COLOR_CLIP_QUANTILE,
) -> float:
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]

    if finite.size == 0:
        return 1.0

    positive = finite[finite > 0]
    if positive.size:
        vmax = float(np.quantile(positive, quantile))
        if vmax <= 0:
            vmax = float(positive.max())
    else:
        vmax = float(np.quantile(finite, quantile))

    if not np.isfinite(vmax) or vmax <= 0:
        vmax = 1.0

    return vmax


def format_colorbar_ticks(colorbar) -> None:
    colorbar.ax.tick_params(labelsize=7, length=2)


def draw_continuous_umap(
    ax,
    coordinates: np.ndarray,
    values: np.ndarray,
    *,
    title: str,
    colorbar_label: str,
    cmap: str,
    vmax: float | None = None,
) -> None:
    values = np.asarray(values, dtype=float)

    ax.scatter(
        coordinates[:, 0],
        coordinates[:, 1],
        s=BACKGROUND_SIZE,
        c=BACKGROUND_COLOR,
        alpha=BACKGROUND_ALPHA,
        linewidths=0,
        edgecolors="none",
        rasterized=RASTERIZE_POINTS,
        zorder=1,
    )

    finite_positive = np.isfinite(values) & (values > 0)
    if finite_positive.any():
        if vmax is None:
            vmax = robust_vmax(values)

        positions = np.flatnonzero(finite_positive)
        positions = positions[
            np.argsort(values[positions], kind="stable")
        ]

        clipped = np.clip(values[positions], 0, vmax)

        scatter = ax.scatter(
            coordinates[positions, 0],
            coordinates[positions, 1],
            s=POINT_SIZE,
            c=clipped,
            cmap=cmap,
            vmin=0,
            vmax=vmax,
            alpha=POINT_ALPHA,
            linewidths=0,
            edgecolors="none",
            rasterized=RASTERIZE_POINTS,
            zorder=2,
        )

        colorbar = ax.figure.colorbar(
            scatter,
            ax=ax,
            fraction=0.045,
            pad=0.02,
        )
        colorbar.set_label(
            colorbar_label,
            fontsize=7.5,
        )
        format_colorbar_ticks(colorbar)
    else:
        ax.text(
            0.5,
            0.5,
            "No positive values",
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=8,
        )

    ax.set_title(title, fontsize=9, pad=5)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")

    for spine in ax.spines.values():
        spine.set_visible(False)


## Load the cNMF model, UMAP, continuous program usages, and Cdh5

In [ ]:
if not H5MU_PATH.exists():
    raise FileNotFoundError(H5MU_PATH)

print("Loading:", H5MU_PATH)
mdata = mu.read_h5mu(H5MU_PATH)

if "cNMF" not in mdata.mod or "rna" not in mdata.mod:
    raise KeyError(
        f"Expected 'cNMF' and 'rna' modalities; found {list(mdata.mod)}"
    )

cnmf = mdata["cNMF"]
rna = mdata["rna"]

if cnmf.n_obs != rna.n_obs:
    raise ValueError("cNMF and RNA modalities have different cell counts.")
if not cnmf.obs_names.equals(rna.obs_names):
    raise ValueError("cNMF and RNA cell order differs.")

scores = dense_array(cnmf.X).astype(float)
if scores.shape != (cnmf.n_obs, cnmf.n_vars):
    raise ValueError(
        f"Unexpected cNMF usage shape {scores.shape}; "
        f"expected {(cnmf.n_obs, cnmf.n_vars)}."
    )
if not np.isfinite(scores).all():
    raise ValueError("cNMF usage matrix contains NaN or infinite values.")

# cNMF usages should be nonnegative. Clip negligible numerical negatives.
minimum_score = float(scores.min())
if minimum_score < -1e-6:
    raise ValueError(
        f"cNMF usage contains substantial negative values: {minimum_score}"
    )
scores = np.maximum(scores, 0.0)

program_ids = infer_program_ids(pd.Index(cnmf.var_names))
program_to_column = {
    program_id: column
    for column, program_id in enumerate(program_ids)
}

missing_programs = [
    program
    for program in SELECTED_PROGRAMS
    if program not in program_to_column
]
if missing_programs:
    raise ValueError(
        f"Selected programs are absent from cNMF: {missing_programs}. "
        f"Detected IDs: {program_ids}"
    )

umap_coordinates, umap_source = resolve_umap(
    mdata=mdata,
    cnmf=cnmf,
    rna=rna,
    prepared_h5ad_path=PREPARED_H5AD_PATH,
)

program_usage = {
    program: scores[:, program_to_column[program]].copy()
    for program in SELECTED_PROGRAMS
}

cdh5_expression, cdh5_label, cdh5_features = extract_gene_expression(
    rna,
    MARKER_GENE,
)

if len(cdh5_expression) != cnmf.n_obs:
    raise ValueError("Cdh5 expression vector has the wrong number of cells.")

cell_table = pd.DataFrame(
    {
        "cell_id": cnmf.obs_names.astype(str),
        "UMAP1": umap_coordinates[:, 0],
        "UMAP2": umap_coordinates[:, 1],
        **{
            f"P{program}_usage": values
            for program, values in program_usage.items()
        },
        "Cdh5_expression": cdh5_expression,
    }
)

print(mdata)
print("UMAP source:", umap_source)
print("Cdh5 source label:", cdh5_label)
print("Matched Cdh5 feature(s):", cdh5_features)

display(
    pd.DataFrame(
        {
            "program": [f"P{x}" for x in SELECTED_PROGRAMS],
            "label": [
                PROGRAM_LABELS[x]
                for x in SELECTED_PROGRAMS
            ],
            "median_usage": [
                float(np.median(program_usage[x]))
                for x in SELECTED_PROGRAMS
            ],
            "positive_cells": [
                int(np.sum(program_usage[x] > 0))
                for x in SELECTED_PROGRAMS
            ],
            "vmax_99th_percentile": [
                robust_vmax(program_usage[x])
                for x in SELECTED_PROGRAMS
            ],
        }
    )
)


## Export cell-level plotting data

In [ ]:
cell_csv = (
    OUTPUT_DIR / f"{OUTPUT_STEM}_cell_level_data.csv.gz"
)
summary_csv = (
    OUTPUT_DIR / f"{OUTPUT_STEM}_plot_scale_summary.csv"
)

cell_table.to_csv(
    cell_csv,
    index=False,
    compression="gzip",
)

summary_rows = []
for program in SELECTED_PROGRAMS:
    values = program_usage[program]
    summary_rows.append(
        {
            "feature": f"P{program}",
            "label": PROGRAM_LABELS[program],
            "minimum": float(np.min(values)),
            "median": float(np.median(values)),
            "mean": float(np.mean(values)),
            "maximum": float(np.max(values)),
            "positive_cells": int(np.sum(values > 0)),
            "color_vmax": robust_vmax(values),
            "color_clip_quantile": COLOR_CLIP_QUANTILE,
        }
    )

summary_rows.append(
    {
        "feature": "Cdh5",
        "label": cdh5_label,
        "minimum": float(np.min(cdh5_expression)),
        "median": float(np.median(cdh5_expression)),
        "mean": float(np.mean(cdh5_expression)),
        "maximum": float(np.max(cdh5_expression)),
        "positive_cells": int(np.sum(cdh5_expression > 0)),
        "color_vmax": robust_vmax(cdh5_expression),
        "color_clip_quantile": COLOR_CLIP_QUANTILE,
    }
)

plot_summary = pd.DataFrame(summary_rows)
plot_summary.to_csv(summary_csv, index=False)

print("Cell-level CSV:", cell_csv)
print("Plot-scale summary:", summary_csv)
display(plot_summary)


## Four separate continuous-usage UMAPs

In [ ]:
program_panel_pdf = (
    OUTPUT_DIR / f"{OUTPUT_STEM}_four_programs.pdf"
)
program_panel_svg = (
    OUTPUT_DIR / f"{OUTPUT_STEM}_four_programs.svg"
)
program_panel_png = (
    OUTPUT_DIR / f"{OUTPUT_STEM}_four_programs.png"
)

fig, axes = plt.subplots(
    2,
    2,
    figsize=PROGRAM_FIGSIZE,
    constrained_layout=True,
)
axes = axes.ravel()

for axis, program in zip(axes, SELECTED_PROGRAMS):
    draw_continuous_umap(
        axis,
        umap_coordinates,
        program_usage[program],
        title=f"P{program} — {PROGRAM_LABELS[program]}",
        colorbar_label=(
            f"P{program} cNMF usage\n"
            f"(clipped at q={COLOR_CLIP_QUANTILE:.2f})"
        ),
        cmap=PROGRAM_CMAP,
        vmax=robust_vmax(program_usage[program]),
    )

fig.suptitle(
    "Continuous usage of prioritized cNMF programs",
    fontsize=11,
    y=1.01,
)

fig.savefig(program_panel_pdf)
fig.savefig(program_panel_svg)
fig.savefig(program_panel_png, dpi=300)
plt.show()
plt.close(fig)

print(program_panel_pdf)
print(program_panel_svg)
print(program_panel_png)


## Save one editable UMAP file per program

In [ ]:
individual_output_rows = []

for program in SELECTED_PROGRAMS:
    fig, ax = plt.subplots(
        figsize=INDIVIDUAL_FIGSIZE,
        constrained_layout=True,
    )

    draw_continuous_umap(
        ax,
        umap_coordinates,
        program_usage[program],
        title=f"P{program} — {PROGRAM_LABELS[program]}",
        colorbar_label=(
            f"P{program} cNMF usage\n"
            f"(clipped at q={COLOR_CLIP_QUANTILE:.2f})"
        ),
        cmap=PROGRAM_CMAP,
        vmax=robust_vmax(program_usage[program]),
    )

    stem = (
        OUTPUT_DIR
        / f"IP_GV_P{program}_continuous_usage_UMAP"
    )
    pdf_path = stem.with_suffix(".pdf")
    svg_path = stem.with_suffix(".svg")
    png_path = stem.with_suffix(".png")

    fig.savefig(pdf_path)
    fig.savefig(svg_path)
    fig.savefig(png_path, dpi=300)
    plt.show()
    plt.close(fig)

    individual_output_rows.append(
        {
            "program": f"P{program}",
            "pdf": str(pdf_path),
            "svg": str(svg_path),
            "png": str(png_path),
        }
    )

display(pd.DataFrame(individual_output_rows))


## Cdh5 marker-gene UMAP

In [ ]:
cdh5_pdf = OUTPUT_DIR / "IP_GV_Cdh5_expression_UMAP.pdf"
cdh5_svg = OUTPUT_DIR / "IP_GV_Cdh5_expression_UMAP.svg"
cdh5_png = OUTPUT_DIR / "IP_GV_Cdh5_expression_UMAP.png"

fig, ax = plt.subplots(
    figsize=GENE_FIGSIZE,
    constrained_layout=True,
)

draw_continuous_umap(
    ax,
    umap_coordinates,
    cdh5_expression,
    title="Cdh5 endothelial marker expression",
    colorbar_label=(
        f"{cdh5_label}\n"
        f"(clipped at q={COLOR_CLIP_QUANTILE:.2f})"
    ),
    cmap=GENE_CMAP,
    vmax=robust_vmax(cdh5_expression),
)

fig.savefig(cdh5_pdf)
fig.savefig(cdh5_svg)
fig.savefig(cdh5_png, dpi=300)
plt.show()
plt.close(fig)

print(cdh5_pdf)
print(cdh5_svg)
print(cdh5_png)


## Five-panel overview: four programs plus Cdh5

In [ ]:
overview_pdf = (
    OUTPUT_DIR / f"{OUTPUT_STEM}_five_panel_overview.pdf"
)
overview_svg = (
    OUTPUT_DIR / f"{OUTPUT_STEM}_five_panel_overview.svg"
)
overview_png = (
    OUTPUT_DIR / f"{OUTPUT_STEM}_five_panel_overview.png"
)

fig, axes = plt.subplots(
    2,
    3,
    figsize=OVERVIEW_FIGSIZE,
    constrained_layout=True,
)
axes = axes.ravel()

for axis, program in zip(axes[:4], SELECTED_PROGRAMS):
    draw_continuous_umap(
        axis,
        umap_coordinates,
        program_usage[program],
        title=f"P{program} — {PROGRAM_LABELS[program]}",
        colorbar_label=f"P{program} cNMF usage",
        cmap=PROGRAM_CMAP,
        vmax=robust_vmax(program_usage[program]),
    )

draw_continuous_umap(
    axes[4],
    umap_coordinates,
    cdh5_expression,
    title="Cdh5 endothelial marker",
    colorbar_label=cdh5_label,
    cmap=GENE_CMAP,
    vmax=robust_vmax(cdh5_expression),
)

axes[5].axis("off")
axes[5].text(
    0.02,
    0.90,
    "Plot notes",
    transform=axes[5].transAxes,
    fontsize=10,
    fontweight="bold",
    va="top",
)
axes[5].text(
    0.02,
    0.78,
    (
        "• Program panels show continuous cNMF usage in all cells.\n"
        "• Cdh5 is log1p normalized when raw counts are available.\n"
        f"• Color is clipped at the {COLOR_CLIP_QUANTILE:.0%} quantile.\n"
        "• Exported CSV values are not clipped.\n"
        "• No program labels are overlaid on cells."
    ),
    transform=axes[5].transAxes,
    fontsize=8.2,
    va="top",
    linespacing=1.5,
)

fig.suptitle(
    "Prioritized endothelial programs and Cdh5 expression",
    fontsize=11,
    y=1.01,
)

fig.savefig(overview_pdf)
fig.savefig(overview_svg)
fig.savefig(overview_png, dpi=300)
plt.show()
plt.close(fig)

print(overview_pdf)
print(overview_svg)
print(overview_png)


## Interpretation

The continuous program-usage UMAPs answer a different question from the original dominant-program map.

- A program can be active in many cells even when it is globally dominant in only a small subset.
- Each panel shows the full distribution of one program across all cells.
- The Cdh5 panel helps determine whether the embedding is broadly endothelial and whether the selected programs occupy Cdh5-positive regions.
- Color scales are calculated separately for each panel, so color intensity should be interpreted within a panel rather than compared numerically between panels.
- The exported CSV contains the unmodified values for quantitative comparisons.